In [1]:
import os
import time
import requests
import pandas as pd

In [ ]:
# CONFIGURATION
CSV_PATH = "../church-fathers.csv"
OUTPUT_BASE = "../data/"

GITHUB_TOKEN = ""  

# GitHub API
PTA_API_BASE = "https://api.github.com/repos/PatristicTextArchive/pta_data/contents/data"
PTA_RAW_BASE = "https://raw.githubusercontent.com/PatristicTextArchive/pta_data/master/data"

# HTTP SESSION
session = requests.Session()
headers = {
    "User-Agent": "pta-downloader/1.0 (+https://example.org)",
    "Accept": "application/vnd.github.v3+json"
}
if GITHUB_TOKEN:
    headers["Authorization"] = f"token {GITHUB_TOKEN}"
session.headers.update(headers)


def get_work_folders(pta_id):
    """Get list of work folders for an author using GitHub API."""
    author_api_url = f"{PTA_API_BASE}/{pta_id}"
    print(f"Fetching work list from API: {author_api_url}")
    
    try:
        r = session.get(author_api_url, timeout=20)
        r.raise_for_status()
        contents = r.json()
        
        # Filter for directories (work folders)
        work_folders = [item['name'] for item in contents if item['type'] == 'dir']
        return work_folders
        
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 404:
            print(f"  Author {pta_id} not found in repository")
        elif e.response.status_code == 403:
            print(f"  Rate limit exceeded. Please add a GitHub token.")
        else:
            print(f"  HTTP Error: {e}")
        return []
    except Exception as e:
        print(f"  Error fetching work folders: {e}")
        return []


def get_xml_files(pta_id, work_id):
    """Get list of Greek edition XML files using GitHub API."""
    work_api_url = f"{PTA_API_BASE}/{pta_id}/{work_id}"
    
    print(f"  Fetching file list from API: {work_api_url}")
    
    try:
        r = session.get(work_api_url, timeout=20)
        r.raise_for_status()
        contents = r.json()
        
        # Filter for Greek edition XML files
        # Greek editions contain: -grc (Greek), -Grc (Greek capitalized)
        # Exclude traslations
        xml_files = [
            item['name'] for item in contents 
            if item['type'] == 'file' 
            and item['name'].endswith('.xml')
            and ('-grc' in item['name'].lower() or '-Grc' in item['name'])  # Greek editions
        ]
        
        return xml_files
        
    except requests.exceptions.HTTPError as e:
        if e.response.status_code == 403:
            print(f"  Rate limit exceeded.")
        else:
            print(f"  Error fetching XML files: {e}")
        return []
    except Exception as e:
        print(f"  Error fetching XML files: {e}")
        return []


def download_xml_file(pta_id, work_id, filename, output_dir):
    """Download a single XML file from GitHub raw content."""
    raw_url = f"{PTA_RAW_BASE}/{pta_id}/{work_id}/{filename}"
    output_path = os.path.join(output_dir, filename)
    
    print(f"    Downloading: {filename}")
    
    try:
        r = session.get(raw_url, timeout=30)
        r.raise_for_status()
        
        with open(output_path, 'wb') as fh:
            fh.write(r.content)
        
        print(f"    ✓ Saved: {output_path}")
        return True
    except Exception as e:
        print(f"    ✗ Download failed: {e}")
        return False


def process_author(cf_id, pta_id, author_name):
    """Process one author: download all their works."""
    author_dir = os.path.join(OUTPUT_BASE, cf_id)
    os.makedirs(author_dir, exist_ok=True)
    
    print(f"\n{'='*60}")
    print(f"Processing {author_name}")
    print(f"CF_ID: {cf_id}, PTA_ID: {pta_id}")
    print(f"{'='*60}")
    
    # Get list of works for this author
    work_folders = get_work_folders(pta_id)
    
    if not work_folders:
        print(f"  No works found for {pta_id} in PTA repository")
        return
    
    print(f"  Found {len(work_folders)} work(s): {', '.join(work_folders)}")
    
    total_downloaded = 0
    
    # Process each work
    for work_id in work_folders:
        print(f"\n  → Work: {work_id}")
        
        # Get XML files for this work
        xml_files = get_xml_files(pta_id, work_id)
        
        if not xml_files:
            print(f"    No Greek edition XML files found for {work_id}")
            continue
        
        print(f"    Found {len(xml_files)} Greek edition file(s)")
        
        # Download each XML file
        for xml_file in xml_files:
            if download_xml_file(pta_id, work_id, xml_file, author_dir):
                total_downloaded += 1
            time.sleep(0.3)  # Be nice to GitHub
        
        time.sleep(0.5)  # Pause between works
    
    print(f"\n✓ Completed {author_name}: {total_downloaded} file(s) saved in {author_dir}")


def main():
    # Load CSV
    df = pd.read_csv(CSV_PATH, sep=';', encoding='utf-8')
    
    # Filter rows with pta_id
    df_with_pta = df[df['pta_id'].notna()].copy()
    print(f"Found {len(df_with_pta)} authors with PTA IDs.\n")
    
    # Process each author
    for _, row in df_with_pta.iterrows():
        cf_id = row['CF_ID']
        pta_id = row['pta_id']
        name = row['Name']
        
        process_author(cf_id, pta_id, name)
        time.sleep(1)  # Pause between authors
    
    print("All downloads complete!")


if __name__ == "__main__":
    main()

Found 9 authors with PTA IDs.


Processing Hippolyt von Rom
CF_ID: CF007, PTA_ID: pta0037
Fetching work list from API: https://api.github.com/repos/PatristicTextArchive/pta_data/contents/data/pta0037
  Found 1 work(s): pta001

  → Work: pta001
  Fetching file list from API: https://api.github.com/repos/PatristicTextArchive/pta_data/contents/data/pta0037/pta001
    Found 1 Greek edition file(s)
    Downloading: pta0037.pta001.pta-grc1.xml
    ✓ Saved: ../data/CF007/pta0037.pta001.pta-grc1.xml

✓ Completed Hippolyt von Rom: 1 file(s) saved in ../data/CF007

Processing Origenes
CF_ID: CF009, PTA_ID: pta0007
Fetching work list from API: https://api.github.com/repos/PatristicTextArchive/pta_data/contents/data/pta0007
  Found 4 work(s): pta007, pta008, pta010, pta011

  → Work: pta007
  Fetching file list from API: https://api.github.com/repos/PatristicTextArchive/pta_data/contents/data/pta0007/pta007
    Found 1 Greek edition file(s)
    Downloading: pta0007.pta007.pta-grc1.xml
    ✓ Saved: